In [5]:
from pathlib import Path

from astropy.io import fits
from astropy.io.fits.fitsrec import FITS_rec
from numpy.typing import NDArray
import numpy as np
from scipy.interpolate import griddata

import darksun as ds

ds.show.set_figures_darkbkg()

## **Energy Bands Absrp Dist Extraction - Analytic Approach**

In [6]:
def load_fits_data(path: Path, ext: int = 1) -> FITS_rec:
    """Loads the FITS file data from chosen extension."""
    return fits.getdata(path, ext=ext, header=False)

def extract_transmission(
    data: FITS_rec,
    energy_range: tuple[float, float] = (2.0, 50.0),
) -> tuple[NDArray, NDArray]:
    """
    Extracts photons transmission values in specified `energy_range`.
    """
    energy: NDArray = data.field(0)
    transmission: NDArray = data.field(1)
    low, high = energy_range
    band: NDArray = (energy >= low) & (energy <= high)
    return energy[band], transmission[band]

def interp(
    x: NDArray,
    y: NDArray,
    energy: NDArray,
) -> NDArray:
    """Interpolates `y` values in given `energy` values."""
    # ...
    # some preprocess (?)
    # ...
    return griddata(x, y, energy, method='linear')

def integrate(arr: NDArray, bins: NDArray) -> float:
    """Integrates input array."""
    return np.cumsum(arr * bins)[-1]

In [7]:
def _transform(theta_x: float, theta_y: float) -> float:
    """
    Computes transformation between local-frame and polar
    frame. Input angles are in [deg]. The output is the
    tangent of the polar angle wrt the XY plane.
    """
    tan_x, tan_y = map(
        lambda x: np.tan(np.deg2rad(x)), (theta_x, theta_y),
    )
    return np.sqrt(tan_x ** 2 + tan_y ** 2)

def compute_theta(theta_x: float, theta_y: float) -> float:
    """
    Computes the polar angular coord wrt to the xy plane.
    Both `theta_x` and `theta_y` are in [deg].
    Output angle value is in [deg].
    """
    xi: float = _transform(theta_x, theta_y)
    return np.rad2deg(np.atan(xi))

def project_absrp_correction(
    distance: float,
    theta_x: float,
    theta_y: float,
) -> NDArray:
    """
    Computes the source local-frame coords correction for the
    detector absorption photons distance in a given energy band.
    """
    xi: float = _transform(theta_x, theta_y)
    theta_plane_proj: float = np.sin(np.atan(xi))
    local_angles: NDArray = np.deg2rad(np.array([theta_x, theta_y]))
    phi_local_proj: NDArray = np.tan(local_angles) / (xi + 1e-8)
    return distance * theta_plane_proj * phi_local_proj

- We want to compute the median absorption distance on the detector for an incident photon with energy $E$, for the SDDs inside each LEM-X camera.

- The travel distance $x$ for a photon with energy $E$ is linked to the detector material transmission (Si-based):

$$ T(x, E) \propto \text{exp}[- \rho\mu(E) \cdot x] $$

- The median distance is then:

$$ \hat{\text{x}}(E) = \frac{\text{ln}(2)}{\rho\mu(E)} $$

In [8]:
settings_path: str = "/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data/Simulations/camera_settings"
#settings_path: str = "/mnt/d/PhD_AASS/Coding/Images_fits/camera_settings"

detSi_matten: FITS_rec = load_fits_data(Path(settings_path, "detectorSi_absrp.fits"))

In [9]:
# extract detector density, thickness and mass attenuation (ON-AXIS)
rho: float = 2.33   # [g/cm3] (from FITS header)
d: float = 0.045    # [cm]
energy, matten = extract_transmission(detSi_matten)  # [keV], [cm2/g]

In [17]:
def compute_median_dist(ph_energy: float) -> float:
    """
    Computes detector median absorption distance
    in [cm] for a photon with `ph_energy` keV.
    """
    mu = interp(energy, matten, ph_energy)
    return np.log(2) / (rho * mu)


ph_energy = 20.0                # [keV]
theta_x, theta_y = 30.0, 30.0

# LoS mean absrp distance in the detector [mm]
dist: float = 10 * compute_median_dist(ph_energy)

_theta = np.deg2rad(compute_theta(theta_x, theta_y))
depth = dist * np.cos(_theta)
print(
    f'Photon mean distance absorption: {dist:.4f} mm\n'
    f'Detector penetration: {depth:.4f} mm (detected: {depth < 10 * d})\n'
    f'Impact deviation: {dist * np.sin(_theta):.4f} mm\n'
)

dsx, dsy = project_absrp_correction(dist, theta_x, theta_y)
print(f'Local-frame coords correction (x, y): {dsx:.4f}, {dsy:.4f} mm')

Photon mean distance absorption: 0.7275 mm
Detector penetration: 0.5636 mm (detected: False)
Impact deviation: 0.4601 mm

Local-frame coords correction (x, y): 0.3254, 0.3254 mm


- If we want to know the median distance in a given energy interval $[E_{1}, E_{2}]$, we have to integrate.

In [19]:
theta_x, theta_y = 15.0, 15.0                      # [deg]
energy_range: tuple[float, float] = (8.0, 14.0)  # [keV]

energy_, matten_ = extract_transmission(detSi_matten, energy_range)  # [keV], [cm2/g]
dist: float = 10 * (np.log(2) / rho) * integrate(1.0 / matten_[:-1], np.diff(energy_)) # [mm]

_theta = np.deg2rad(compute_theta(theta_x, theta_y))
depth = dist * np.cos(_theta)
print(
    f'Photon mean distance absorption: {dist:.4f} mm\n'
    f'Detector penetration: {depth:.4f} mm (detected: {depth < 10 * d})\n'
    f'Impact deviation: {dist * np.sin(_theta):.4f} mm\n'
)

dsx, dsy = project_absrp_correction(dist, theta_x, theta_y)
print(f'Local-frame coords correction (x, y): {dsx:.4f}, {dsy:.4f} mm')

Photon mean distance absorption: 0.7568 mm
Detector penetration: 0.7077 mm (detected: False)
Impact deviation: 0.2682 mm

Local-frame coords correction (x, y): 0.1896, 0.1896 mm


<br>

<br>

## **IROS Benchmark Test - Energy Bands Absrp Dist Extraction**

- In this approach, we'll compute the **median absorption distance** of the photons in a given energy band $\Delta$E directly from the WISEMAN originated events.

- In the "IROS benchmark" test, a mock sky-field with Crab-like sources is simulated. The sources are uniformly distribuited in the camera FoV, and have the same emission spectrum (from the Crab pulsar, i.e. with breaking index $ n = 2.1 $, typical of a stable neutron star with energy loss dominated by EM emission).

- In the `demo_source_obs_spectrum.ipynb` file it is established that Crab-like sources have $ \sim $ stable photons emission percentage wrt total emission counts in given energy bands and at different sky coords. To fit well enough a source during the sky-field IROS reconstruction, we can decompose the the full band ([2, 50] keV) source shadowgram into $\Delta\textit{E}$ related components, weighted by the counts percentage in that energy band wrt the measured total counts.

- Since photons with higher energies are absorbed deeper in the camera SDD, the shadowgram components have energy dependent coordinates. By measuring the median absrp distance in the chosen energy bands from the WM events, we can estimate a correction $\delta\textit{s}$ to apply to the source coords in the camera local-frame. The correction depends on the photons energy and on the source coords.

- By analysing the median absrp distance $\hat{\xi}$ of the photons in a given energy band $\Delta\textit{E}$ wrt the source coords in the camera local-frame, maybe it'll be possible to extrapolate a law to compute the correction $\delta\textit{s}$.

In [10]:
from singleCAM_IROS._pipeline_support import _handle_dirpaths

from pathlib import Path
import numpy as np
from numpy.typing import NDArray
from astropy.io.fits.fitsrec import FITS_rec

from bloodmoon.types import CoordEquatorial
from bloodmoon.io import simulation_files
from bloodmoon.mask import CodedMaskCamera
from bloodmoon.mask import codedmask, count

import darksun as ds
from darksun.data import DataLoader, CatalogueLoader
from darksun.benchmarking import source_catalogue_data

ds.show.set_figures_darkbkg()

**Mask and Data Specifics**

In [2]:
MASK_FITS: str = "wfm_mask_NTHT_20250725.fits"

SKYFIELD: str = "IROSDummy"
DATA_FITS: str = "iros_benchmark_2-50keV_mask_050_1040x17_1ks"

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "reconstructed"

UPS_X: int = 5
UPS_Y: int = 1

VIGNETTING: bool = True
PSFY: bool = True

**Load Mask and Data**

In [4]:
# load filepaths
mask_path, simul_data, save_path = _handle_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
)
wfm: CodedMaskCamera = codedmask(mask_path, UPS_X, UPS_Y)
filepaths: dict[str, dict[str, Path]] = simulation_files(simul_data)

# data from camera A
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET])
catalogueA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])

# data from camera B
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET])
catalogueB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

In [22]:
type EnergyRange = tuple[float, float]

def get_source_coords(
    sourceID: str,
    catalogue: CatalogueLoader,
) -> CoordEquatorial:
    """Retrieves the source RA/Dec coords from catalogue."""
    data = source_catalogue_data(sourceID, catalogue.DLdata)
    return CoordEquatorial(data['RA'], data['DEC'])

def extract_photons(
    sdl: DataLoader,
    coords: CoordEquatorial,
) -> FITS_rec:
    """Extracts photons from input direction."""
    return ds.select_source_photons(coords, sdl.DLdata, False)

def filter_energy(
    photons: FITS_rec,
    E_min: float | None,
    E_max: float | None,
) -> FITS_rec:
    """Filters the input photons by energy, in [keV]."""
    return ds.filter_data(photons, E_min=E_min, E_max=E_max, coords=None)



sourceID: str = 's17'
eband: EnergyRange = (2.0, 4.0)

coords: CoordEquatorial = get_source_coords(sourceID, catalogueA)
phs: FITS_rec = extract_photons(sdlA, coords)
eband_phs: FITS_rec = filter_energy(phs, *eband)

len(sdlA.DLdata), len(phs), len(eband_phs)

(3561427, 906779, 292061)

In [30]:
m = (eband_phs['ZPHOTON'] > 0)
np.max(eband_phs['ZPHOTON'][m]), np.std(eband_phs['ZPHOTON'][m])

(np.float64(0.22499998739643415), np.float64(0.03569190144531754))